# Gomoku Agent — Demo Notebook

**Course Project Demo** | Gomoku (Five-in-a-Row) AI Agent with Minimax + Alpha-Beta Pruning

This notebook demonstrates:
1. [Download & Install](#1-download--install)
2. [Run the App (Interactive Game)](#2-run-the-app-interactive-game)
3. [Run Test Scripts](#3-run-test-scripts)
4. [Experiment 1 — Performance Metrics](#4-experiment-1--performance-metrics)
5. [Experiment 2 — AI vs AI Benchmark](#5-experiment-2--ai-vs-ai-benchmark-benchmarkpy)
6. [Experiment 3 — Minimax vs Alpha-Beta](#6-experiment-3--minimax-vs-alpha-beta-benchmark_algopy)

> **Environment:** Designed to run on [Google Colab](https://colab.research.google.com/). The interactive GUI (`main.py`) requires a local desktop; all other sections run headlessly in Colab.

---
## 1. Download & Install

Clone the repository from GitHub and install all required dependencies.

In [ ]:
# ── Clone repository ──────────────────────────────────────────────────────────
!git clone https://github.com/YaruG1022/Gomoku-Agent.git
%cd Gomoku-Agent

In [ ]:
# ── Install dependencies ──────────────────────────────────────────────────────
# numpy  : numerical arrays used by the heuristic evaluation
# pygame : desktop GUI (not used in Colab, but installed to satisfy imports)
# pytest : test runner
!pip install -r requirements.txt -q
print("All dependencies installed.")

In [ ]:
# ── Verify installation ───────────────────────────────────────────────────────
import importlib, sys

required = ["numpy", "pygame", "pytest"]
for pkg in required:
    mod = importlib.import_module(pkg)
    ver = getattr(mod, "__version__", "(no version)")
    print(f"  {pkg:10s} {ver}")

print(f"\nPython {sys.version}")

---
## 2. Run the App (Interactive Game)

> ⚠️ **This notebook cannot run the full interactive app.** The pygame GUI requires a local desktop environment. The cell below is **for demonstration only** — it shows the AI search logic running in text mode with pre-scripted moves, not a playable game.

**To play the full app, run it locally:**
```bash
python main.py
```

### Full app features (local only)

**On launch — Mode selection menu:**

| Mode | Description |
|---|---|
| **Player vs AI** | You play as Black; AI (White, depth-3 Alpha-Beta) responds automatically |
| **AI vs AI** | Launches `benchmark.py` as a subprocess for automated games |

**During Player vs AI:**

| Action | How |
|---|---|
| Place a stone | Left-click a grid intersection |
| Undo last full turn (your move + AI move) | Click **Undo** button · or press **U** / **Backspace** |
| Restart game | Click **Restart** button · or press **R** / **Escape** |

The status bar shows whose turn it is and announces the winner. The last move is highlighted with a red dot.

---

### Notebook demo (headless)

The cell below demonstrates the **same AI search logic** (depth-3 Alpha-Beta) in text mode: 5 pre-scripted human moves are played and the board state is printed after each move.

In [ ]:
# ── Headless AI demo: Human (Black) vs AI (White) ────────────────────────────
from gomoku.board import BLACK, WHITE, EMPTY
from gomoku.game import Game
from gomoku.search import alphabeta, reset_counters, reset_game_state

AI_DEPTH = 3
STONE = {EMPTY: ".", BLACK: "●", WHITE: "○"}


def print_board(board, last_move=None):
    """Pretty-print the 15×15 board with coordinates."""
    cols = "  " + " ".join(f"{c:2d}" for c in range(board.size))
    print(cols)
    for r in range(board.size):
        row_str = ""
        for c in range(board.size):
            cell = STONE[board.grid[r][c]]
            if last_move and (r, c) == last_move:
                cell = f"\033[91m{cell}\033[0m"  # highlight last move in red
            row_str += f"  {cell}"
        print(f"{r:2d}{row_str}")
    print()


def ai_move(game, player, depth=AI_DEPTH):
    reset_counters()
    _, move = alphabeta(
        game.board, depth=depth,
        alpha=float("-inf"), beta=float("inf"),
        maximizing_player=True, player=player,
    )
    return move


# Pre-scripted human moves — simulating a typical opening
human_moves = [(7, 7), (7, 8), (8, 8), (6, 9), (9, 9)]

reset_game_state()
game = Game()

print("=" * 40)
print(" Human (●Black) vs AI (○White)")
print(f" AI search depth: {AI_DEPTH}")
print("=" * 40)
print_board(game.board)

move_num = 0
for human_move in human_moves:
    if game.is_over():
        break

    # ── Human move (Black) ────────────────────────
    r, c = human_move
    game.make_move(r, c)
    move_num += 1
    print(f"Move {move_num:2d} — Human  (●) plays ({r}, {c})")
    print_board(game.board, last_move=(r, c))

    if game.is_over():
        break

    # ── AI move (White) ───────────────────────────
    import time
    t0 = time.perf_counter()
    ai_r, ai_c = ai_move(game, WHITE)
    dt = time.perf_counter() - t0
    game.make_move(ai_r, ai_c)
    move_num += 1
    print(f"Move {move_num:2d} — AI     (○) plays ({ai_r}, {ai_c})  [{dt:.3f}s]")
    print_board(game.board, last_move=(ai_r, ai_c))

# ── Result ────────────────────────────────────────
if game.is_over():
    winner = game.get_winner()
    if winner == BLACK:
        print("Result: Human (●Black) wins!")
    elif winner == WHITE:
        print("Result: AI (○White) wins!")
    else:
        print("Result: Draw.")
else:
    print("Game still in progress after demo moves.")

---
## 3. Run Test Scripts

The project ships with four unit-test modules covering the board, heuristic, move generation, and search components.

| File | What it tests |
|---|---|
| `tests/test_board.py` | Board construction, stone placement, win/draw detection |
| `tests/test_heuristic.py` | Pattern scoring, weight overrides |
| `tests/test_move_gen.py` | Candidate-move generation near existing stones |
| `tests/test_search.py` | Minimax / Alpha-Beta correctness and agreement |

In [ ]:
# ── Run the full test suite ───────────────────────────────────────────────────
!pytest tests/ -v

In [ ]:
# ── Run a single test module ──────────────────────────────────────────────────
!pytest tests/test_search.py -v

---
## 4. Experiment 1 — Performance Metrics

**Script:** `performance_metrics.py`

Measures three fundamental AI performance indicators:

| Metric | Description |
|---|---|
| **Decision Time** | Average seconds the AI takes to pick a move at depth 1-3 |
| **Win Rate** | How often the AI (depth-2) beats a random-move player over 100 games |
| **Game Length** | Average number of moves per game (AI depth-2 vs random) |

In [ ]:
# ── Metric 1: AI decision time per move at depth 1 / 2 / 3 ──────────────────
import performance_metrics
import random
random.seed(42)

performance_metrics.measure_decision_time()

In [ ]:
# ── Metric 2: Win rate — AI (depth-2) vs random player over 100 games ────────
performance_metrics.measure_win_rate(num_games=100, depth=2)

In [ ]:
# ── Metric 3: Average game length — AI (depth-2) vs random over 100 games ────
performance_metrics.measure_game_length(num_games=100, depth=2)

---
## 5. Experiment 2 — AI vs AI Benchmark (`benchmark.py`)

**Script:** `benchmark.py`

Runs fully automated AI-vs-AI games across all combinations of Black depth × White depth and reports:

- **Win-rate matrix** — which depth wins against which
- **Search efficiency** — nodes explored, pruning rate, effective branching factor (EBF), average time per move

Results are saved to `results/benchmark/benchmark_<run_id>.txt` (human-readable) and `.csv` (machine-readable).

### 5a. Quick run (depths 1–2, 2 games per matchup)

In [ ]:
# ── Quick run: depths 1-2, 2 games per matchup ───────────────────────────────
# 4 matchups × 2 games = 8 games total; fast enough to verify everything works.
!python benchmark.py --depths 1 2 --games 2

In [ ]:
# ── Display the human-readable result file ───────────────────────────────────
import glob, os

txt_files = sorted(glob.glob("results/benchmark/benchmark_*.txt"))
if txt_files:
    print(f"Result file: {txt_files[-1]}\n")
    with open(txt_files[-1]) as f:
        print(f.read())
else:
    print("No result file found — run the cell above first.")

### 5b. Full reproduction run (depths 1–3, 3 games per matchup)

> **Expected wall time:** ~5–15 min on Colab (depth-3 vs depth-3 is the slowest matchup).
> 9 matchups × 3 games = 27 games total.

In [ ]:
# ── Full reproduction: depths 1-3, 3 games/matchup ───────────────────────────
# 9 matchups × 3 games = 27 games.  Reproduces the paper's Table results.
!python benchmark.py --depths 1 2 3 --games 3

In [ ]:
# ── Analyze: win-rate matrix & efficiency summary ─────────────────────────────
import glob

csv_files = sorted(glob.glob("results/benchmark/benchmark_*.csv"))
latest_csv = csv_files[-1]
print(f"Analyzing: {latest_csv}\n")
!python analyze.py --csv {latest_csv}

---
## 6. Experiment 3 — Minimax vs Alpha-Beta (`benchmark_algo.py`)

**Script:** `benchmark_algo.py`

Compares **plain Minimax** against **Alpha-Beta + TT + Killer + History** at the same search depth. Both sides search identically deep so the only variable is the algorithm.

Key metrics reported:

| Metric | Description |
|---|---|
| **Node reduction** | % fewer nodes Alpha-Beta explores vs Minimax |
| **Pruning rate** | Fraction of subtrees cut by α-β; higher = better |
| **Speedup** | How many times faster Alpha-Beta is per move |
| **EBF** | Effective Branching Factor — ideal α-β approaches $\sqrt{b}$ of Minimax |

Results are saved to `results/benchmark_algo/`.

### 6a. Quick run (depths 1–2, 2 games each)

In [ ]:
# ── Quick run: depths 1-2, 2 games each ──────────────────────────────────────
# Alternates which colour plays Minimax so results are symmetric.
!python benchmark_algo.py --depths 1 2 --games 2

In [ ]:
# ── Display the algo benchmark result file ───────────────────────────────────
import glob

txt_files = sorted(glob.glob("results/benchmark_algo/benchmark_algo_*.txt"))
if txt_files:
    print(f"Result file: {txt_files[-1]}\n")
    with open(txt_files[-1]) as f:
        print(f.read())
else:
    print("No result file found — run the cell above first.")

### 6b. Full reproduction run + plots (depths 1–3, 3 games each)

> **Expected wall time:** ~5–10 min on Colab.
> 3 depths × 3 games = 9 games, alternating which side plays Minimax.

In [ ]:
# ── Full reproduction: depths 1-3, 3 games each ──────────────────────────────
!python benchmark_algo.py --depths 1 2 3 --games 3

In [ ]:
# ── Visualize: Minimax vs Alpha-Beta comparison plots ─────────────────────────
%matplotlib inline
import glob, sys

csv_files = sorted(glob.glob("results/benchmark_algo/benchmark_algo_*.csv"))
latest_csv = csv_files[-1]
print(f"Plotting: {latest_csv}\n")

sys.argv = ["analyze_algo.py", latest_csv]
import analyze_algo

rows = analyze_algo.load(latest_csv)
data = analyze_algo.aggregate_by_depth(rows)
print(f"Depths: {sorted(data.keys())}  "
      f"({sum(v['n_games'] for v in data.values())} games total)\n")
analyze_algo.plot(data, latest_csv)

In [ ]:
# ── Summary table: key metrics at each depth ──────────────────────────────────
import pandas as pd

rows_summary = []
for depth, d in sorted(data.items()):
    rows_summary.append({
        "Depth":          depth,
        "MM avg nodes":   f"{d['mm_avg_nodes']:,.0f}",
        "AB avg nodes":   f"{d['ab_avg_nodes']:,.0f}",
        "Node reduction": f"{d['node_red_pct']:.1f}%",
        "Pruning rate":   f"{d['ab_pruning_rate']*100:.1f}%",
        "Speedup":        f"{d['speedup']:.2f}×",
        "MM EBF":         f"{d['mm_avg_ebf']:.2f}",
        "AB EBF":         f"{d['ab_avg_ebf']:.2f}",
    })

df = pd.DataFrame(rows_summary).set_index("Depth")
print("=== Minimax vs Alpha-Beta — Key Metrics ===")
print(df.to_string())

---
### Interpreting the Results

| Metric | What it means |
|---|---|
| **Node reduction** | % fewer nodes Alpha-Beta explores vs. plain Minimax at the same depth |
| **Pruning rate** | Fraction of subtrees cut by α-β — higher is better |
| **Speedup** | How many times faster Alpha-Beta is per move |
| **EBF** | Effective Branching Factor — how many children the search effectively considers per node; ideal α-β approaches $\sqrt{b}$ of Minimax |

**Expected trend:** node reduction and speedup increase with depth, confirming that Alpha-Beta's advantage grows as the search tree deepens.